# Scanner de Temas Emergentes v8

| Celda | Que hace | Cuando |
|-------|----------|--------|
| 1 | Instala librerias | Solo la primera vez |
| 2 | Carga el codigo | SIEMPRE antes de 3 o 4 |
| 3 | Watchlist personal | ~30 seg |
| 4 | S&P 500 completo | ~4 min |

Novedades v8: precios, max 52s, rendimientos, rangos de entrada, analisis Claude

In [ ]:
!pip install yfinance pandas requests beautifulsoup4 anthropic -q
print('OK librerias instaladas')

In [ ]:
# CELDA 2 — Ejecutar SIEMPRE antes de la 3 o la 4
import math, warnings, json, base64
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import yfinance as yf
warnings.filterwarnings('ignore')

GITHUB_USER    = 'Carolo-III'
GITHUB_REPO    = 'scanner-temas'
GITHUB_TOKEN   = ''   # <-- Token GitHub
ANTHROPIC_KEY  = ''   # <-- API key Anthropic (https://console.anthropic.com)

PERSONAL_WATCHLIST = {
    'Semiconductores': ['ARM','AMD','MU','MTSI','POET','SMCI'],
    'Infraestructura AI': ['ANET','VRT','APLD','CORZ','IREN','CIFR','CRWV'],
    'Espacio y Defensa': ['RKLB','LUNR','ASTS','KTOS','BWXT'],
    'Cuantica': ['IONQ'],
    'Robotica AI': ['BBAI','TSLA'],
    'Crypto Fintech': ['RDDT'],
    'Minerales': ['MP','UAMY'],
    'Biotech': ['VKTX','ACRV','IBRX'],
    'Hardware': ['SNDK'],
    'Momentum': ['KOPN','ONDS','BE','LITE','GILT'],
}

SECTOR_LABELS = {
    'Technology':'Tecnologia','Health Care':'Salud','Financials':'Finanzas',
    'Consumer Discretionary':'Consumo Discrecional','Industrials':'Industriales',
    'Communication Services':'Comunicacion','Consumer Staples':'Consumo Basico',
    'Energy':'Energia','Utilities':'Utilities','Real Estate':'Real Estate','Materials':'Materiales',
}

def download_prices(tickers, period='1y'):
    if not tickers:
        return pd.DataFrame(), pd.DataFrame()
    LOTE = 50
    all_c, all_v, all_h, all_l = [], [], [], []
    lotes = [tickers[i:i+LOTE] for i in range(0, len(tickers), LOTE)]
    for i, lote in enumerate(lotes):
        if len(lotes) > 1:
            print('    Lote ' + str(i+1) + '/' + str(len(lotes)) + '...', end=' ')
        try:
            raw = yf.download(lote, period=period, interval='1d',
                              auto_adjust=True, progress=False, threads=True)
            if raw.empty:
                if len(lotes) > 1: print('vacio')
                continue
            if isinstance(raw.columns, pd.MultiIndex):
                c = raw['Close']; v = raw['Volume']
                h = raw['High'];  l = raw['Low']
            else:
                c = raw[['Close']]; c.columns = lote[:1]
                v = raw[['Volume']]; v.columns = lote[:1]
                h = raw[['High']];  h.columns = lote[:1]
                l = raw[['Low']];   l.columns = lote[:1]
            all_c.append(c); all_v.append(v)
            all_h.append(h); all_l.append(l)
            if len(lotes) > 1: print('OK')
        except Exception as e:
            if len(lotes) > 1: print('error: ' + str(e))
    if not all_c:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    return (pd.concat(all_c, axis=1), pd.concat(all_v, axis=1),
            pd.concat(all_h, axis=1), pd.concat(all_l, axis=1))

def rs_score(t, b, w):
    try:
        t, b = t.dropna(), b.dropna()
        if len(t) < w or len(b) < w: return None
        return round((t.iloc[-1]/t.iloc[-w]-1)*100 - (b.iloc[-1]/b.iloc[-w]-1)*100, 2)
    except: return None

def volume_zscore(v, w=20):
    try:
        v = v.dropna()
        if len(v) < w+1: return None
        m, s = v.iloc[-(w+1):-1].mean(), v.iloc[-(w+1):-1].std()
        return round((v.iloc[-1]-m)/s, 2) if s else 0.0
    except: return None

def detect_breakout(p, v, lb=50, vm=1.4):
    try:
        p, v = p.dropna(), v.dropna()
        if len(p) < lb: return {'breakout': False, 'days_ago': None, 'breakout_level': None}
        for d in range(1, 11):
            bh  = p.iloc[-(lb+d):-(d+5)].max()
            bvm = v.iloc[-(lb+d):-(d+5)].mean()
            if p.iloc[-d] > bh and v.iloc[-d] > vm*bvm:
                return {'breakout': True, 'days_ago': d, 'breakout_level': round(float(bh), 2)}
        # Nivel de ruptura potencial aunque no haya roto aun
        bh = p.iloc[-lb:-5].max()
        return {'breakout': False, 'days_ago': None, 'breakout_level': round(float(bh), 2)}
    except: return {'breakout': False, 'days_ago': None, 'breakout_level': None}

def ma_health(p):
    try:
        p = p.dropna()
        result = {'ma20': False, 'ma50': False, 'ma200': False,
                  'ma20_val': None, 'ma50_val': None, 'ma200_val': None}
        c = float(p.iloc[-1])
        if len(p) >= 20:
            ma20 = float(p.rolling(20).mean().iloc[-1])
            result['ma20'] = bool(c > ma20)
            result['ma20_val'] = round(ma20, 2)
            result['pct_ma20'] = round((c/ma20 - 1)*100, 1)
        if len(p) >= 50:
            ma50 = float(p.rolling(50).mean().iloc[-1])
            result['ma50'] = bool(c > ma50)
            result['ma50_val'] = round(ma50, 2)
            result['pct_ma50'] = round((c/ma50 - 1)*100, 1)
        if len(p) >= 200:
            ma200 = float(p.rolling(200).mean().iloc[-1])
            result['ma200'] = bool(c > ma200)
            result['ma200_val'] = round(ma200, 2)
            result['pct_ma200'] = round((c/ma200 - 1)*100, 1)
        return result
    except: return {'ma20': False, 'ma50': False, 'ma200': False}

def price_stats(p, h, v_series):
    try:
        p = p.dropna()
        if len(p) < 5: return {}
        current = float(p.iloc[-1])
        # Max y min 52 semanas (252 sesiones)
        w52 = min(252, len(p))
        high52 = float(p.iloc[-w52:].max())
        low52  = float(p.iloc[-w52:].min())
        pct_from_high = round((current/high52 - 1)*100, 1)
        pct_from_low  = round((current/low52  - 1)*100, 1)
        # Rendimientos
        ret_1w  = round((current/p.iloc[-5]  - 1)*100, 1) if len(p) >= 5  else None
        ret_1m  = round((current/p.iloc[-20] - 1)*100, 1) if len(p) >= 20 else None
        ret_3m  = round((current/p.iloc[-65] - 1)*100, 1) if len(p) >= 65 else None
        # Minimo 5 dias para stop
        low5 = float(p.iloc[-5:].min()) if len(p) >= 5 else current * 0.95
        return {
            'price': round(current, 2),
            'high52': round(high52, 2),
            'low52':  round(low52, 2),
            'pct_from_high': pct_from_high,
            'pct_from_low':  pct_from_low,
            'ret_1w': ret_1w, 'ret_1m': ret_1m, 'ret_3m': ret_3m,
            'low5': round(low5, 2),
        }
    except: return {}

def calc_entry_range(ps, bi, mah):
    try:
        price   = ps.get('price')
        if not price: return {}
        low5    = ps.get('low5', price * 0.95)
        high52  = ps.get('high52', price)
        ma50    = mah.get('ma50_val')
        ma200   = mah.get('ma200_val')

        if bi.get('breakout'):
            # En ruptura activa
            bl = bi.get('breakout_level', price * 0.95)
            entry_lo = round(bl * 1.000, 2)
            entry_hi = round(bl * 1.030, 2)
            stop     = round(max(low5, bl * 0.930), 2)
            # Objetivo: proyectar la altura de la base
            base_height = price - bl
            target = round(price + base_height, 2)
            rr = round((target - entry_hi) / (entry_hi - stop), 1) if (entry_hi - stop) > 0 else None
            return {
                'tipo': 'Ruptura activa',
                'entry_lo': entry_lo, 'entry_hi': entry_hi,
                'stop': stop, 'target': target, 'rr': rr,
            }
        elif ma50 and abs(price/ma50 - 1) < 0.04:
            # Pullback a MM50
            entry_lo = round(ma50 * 0.98, 2)
            entry_hi = round(ma50 * 1.02, 2)
            stop     = round(ma50 * 0.95, 2)
            target   = round(high52, 2)
            rr = round((target - entry_hi) / (entry_hi - stop), 1) if (entry_hi - stop) > 0 else None
            return {
                'tipo': 'Pullback MM50',
                'entry_lo': entry_lo, 'entry_hi': entry_hi,
                'stop': stop, 'target': target, 'rr': rr,
            }
        elif bi.get('breakout_level'):
            # En base, aun sin romper
            bl = bi.get('breakout_level')
            return {
                'tipo': 'En vigilancia',
                'entry_lo': None, 'entry_hi': None,
                'stop': None, 'target': None, 'rr': None,
                'nivel_ruptura': round(bl, 2),
            }
        return {'tipo': 'Sin setup', 'entry_lo': None, 'entry_hi': None, 'stop': None, 'target': None, 'rr': None}
    except: return {}

def composite_score(r4, r13, vz, bo, ma):
    s = 0
    if r4  is not None: s += 30 * min(1, max(0, (r4+30)/60))
    if r13 is not None: s += 20 * min(1, max(0, (r13+50)/100))
    if vz  is not None: s += 25 * min(1, max(0, (vz+1)/4))
    if bo: s += 15
    if ma: s += 10 * (sum([ma.get('ma20',False), ma.get('ma50',False), ma.get('ma200',False)])/3)
    return round(s, 1)

def analyze_universe(grps, bench, close_df, vol_df, high_df=None, low_df=None):
    res = []
    for gn, tickers in grps.items():
        for tk in tickers:
            if tk not in close_df.columns: continue
            p   = close_df[tk]
            v   = vol_df[tk] if tk in vol_df.columns else pd.Series(dtype=float)
            h   = high_df[tk] if high_df is not None and tk in high_df.columns else p
            l   = low_df[tk]  if low_df  is not None and tk in low_df.columns  else p
            r4  = rs_score(p, bench, 20)
            r13 = rs_score(p, bench, 65)
            vz  = volume_zscore(v) if not v.empty else None
            bi  = detect_breakout(p, v) if not v.empty else {'breakout':False,'days_ago':None,'breakout_level':None}
            mah = ma_health(p)
            ps  = price_stats(p, h, v)
            er  = calc_entry_range(ps, bi, mah)
            sc  = composite_score(r4, r13, vz, bi['breakout'], mah)
            res.append({
                'ticker': tk, 'group': gn,
                'rs_4w': r4, 'rs_13w': r13, 'vol_z': vz,
                'breakout': bi['breakout'], 'days_ago': bi['days_ago'],
                'breakout_level': bi.get('breakout_level'),
                'ma20': mah.get('ma20', False),
                'ma50': mah.get('ma50', False),
                'ma200': mah.get('ma200', False),
                'pct_ma50':  mah.get('pct_ma50'),
                'pct_ma200': mah.get('pct_ma200'),
                'ma50_val':  mah.get('ma50_val'),
                'ma200_val': mah.get('ma200_val'),
                'price':        ps.get('price'),
                'high52':       ps.get('high52'),
                'low52':        ps.get('low52'),
                'pct_from_high': ps.get('pct_from_high'),
                'ret_1w': ps.get('ret_1w'),
                'ret_1m': ps.get('ret_1m'),
                'ret_3m': ps.get('ret_3m'),
                'entry_range': er,
                'score': sc,
            })
    return res

def calc_groups(res, is_sp=False):
    gs = {}
    for r in res:
        gs.setdefault(r['group'], []).append(r)
    out = []
    for gn, mb in gs.items():
        scores = [m['score'] for m in mb if m['score'] is not None]
        r4s    = [m['rs_4w'] for m in mb if m['rs_4w'] is not None]
        mb_sorted = sorted(mb, key=lambda x: x['score'] or 0, reverse=True)
        out.append({
            'group':    gn,
            'score':    round(np.mean(scores), 1) if scores else 0,
            'rs_mean':  round(np.mean(r4s), 1) if r4s else 0,
            'breakouts': sum(1 for m in mb if m['breakout']),
            'n':        len(mb),
            'is_sp':    is_sp,
            'top3':     mb_sorted[:3],
            'members':  mb_sorted,
        })
    return sorted(out, key=lambda x: x['score'], reverse=True)

def generate_analysis(data, anthropic_key):
    import anthropic as ant
    client = ant.Anthropic(api_key=anthropic_key)

    groups  = data['groups']
    values  = data['values']
    ts      = data['timestamp']
    mode    = data['mode']

    strong   = [g for g in groups if g['score'] >= 70]
    emerging = [g for g in groups if 50 <= g['score'] < 70]
    weak     = [g for g in groups if g['score'] < 30]
    breakouts = [v for v in values if v['breakout'] and v.get('entry_range')]

    # Construir resumen de datos para Claude
    summary = 'DATOS DEL SCANNER — ' + ts + ' (' + mode + ')\n\n'
    summary += 'TEMAS FUERTES (score >= 70):\n'
    for g in strong[:5]:
        leaders = ', '.join(m['ticker'] + '(' + str(m['score']) + ')' for m in g['top3'])
        summary += '- ' + g['group'] + ': score ' + str(g['score']) + ' | RS: ' + str(g['rs_mean']) + '% | Lideres: ' + leaders + '\n'

    summary += '\nTEMAS EMERGENTES (50-69):\n'
    for g in emerging[:5]:
        leaders = ', '.join(m['ticker'] + '(' + str(m['score']) + ')' for m in g['top3'])
        summary += '- ' + g['group'] + ': score ' + str(g['score']) + ' | RS: ' + str(g['rs_mean']) + '%\n'

    summary += '\nVALORES CON RUPTURA ACTIVA Y SETUP DE ENTRADA:\n'
    for v in breakouts[:8]:
        er = v['entry_range']
        line = '- ' + v['ticker'] + ' (' + v['group'] + '): precio $' + str(v.get('price','?'))
        line += ' | RS4s: ' + str(v.get('rs_4w','?')) + '%'
        line += ' | ' + str(v.get('pct_from_high','?')) + '% desde max 52s'
        if er.get('entry_lo'):
            line += ' | Entrada: $' + str(er['entry_lo']) + '-$' + str(er['entry_hi'])
            line += ' | Stop: $' + str(er['stop'])
            line += ' | Objetivo: $' + str(er['target'])
            if er.get('rr'): line += ' | R/B: 1:' + str(er['rr'])
        summary += line + '\n'

    summary += '\nRESUMEN GENERAL:\n'
    summary += '- Total temas: ' + str(len(groups)) + '\n'
    summary += '- Fuertes (>=70): ' + str(len(strong)) + '\n'
    summary += '- Emergentes (50-69): ' + str(len(emerging)) + '\n'
    summary += '- Sin momentum (<30): ' + str(len(weak)) + '\n'
    summary += '- Rupturas activas: ' + str(len(breakouts)) + '\n'

    prompt = (
        'Eres un analista tecnico de mercados especializado en momentum y analisis sectorial. '
        'Analiza estos datos del scanner de temas emergentes y genera un informe ejecutivo en espanol. '
        'El informe debe ser directo, sin rodeos, orientado a la accion. '
        'Estructura: '
        '1. SITUACION DEL MERCADO (2-3 frases sobre el estado general del momentum) '
        '2. TEMAS PRIORITARIOS (los 3 temas mas fuertes con una frase de contexto cada uno) '
        '3. ALERTAS DE ENTRADA (los 3-5 mejores setups con el rango de entrada, stop y objetivo en formato claro) '
        '4. TEMAS A EVITAR (sectores sin momentum con una frase breve) '
        '5. CONCLUSION (1 frase accionable para hoy) '
        'Usa lenguaje tecnico pero comprensible. Maximo 400 palabras. '
        'IMPORTANTE: incluye siempre este aviso al final: '
        '"Aviso: Este analisis es orientativo y no constituye asesoramiento financiero. '
        'Las decisiones de inversion son responsabilidad exclusiva del inversor."\n\n'
        + summary
    )

    print('Generando analisis con Claude...')
    msg = client.messages.create(
        model='claude-sonnet-4-20250514',
        max_tokens=1000,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return msg.content[0].text

def get_github_file(filename, token, user, repo):
    url = 'https://api.github.com/repos/' + user + '/' + repo + '/contents/' + filename
    headers = {'Authorization': 'token ' + token}
    r = requests.get(url, headers=headers)
    if r.status_code == 200:
        content = base64.b64decode(r.json()['content']).decode('utf-8')
        return json.loads(content), r.json()['sha']
    return None, None

def upload_to_github(filename, content, token, user, repo):
    url = 'https://api.github.com/repos/' + user + '/' + repo + '/contents/' + filename
    headers = {'Authorization': 'token ' + token, 'Content-Type': 'application/json'}
    r = requests.get(url, headers=headers)
    sha = r.json().get('sha') if r.status_code == 200 else None
    if isinstance(content, str):
        content_b64 = base64.b64encode(content.encode('utf-8')).decode('utf-8')
    else:
        content_b64 = base64.b64encode(content).decode('utf-8')
    payload = {'message': 'Actualizar ' + filename + ' - ' + datetime.now().strftime('%d/%m/%Y %H:%M'), 'content': content_b64}
    if sha: payload['sha'] = sha
    r = requests.put(url, headers=headers, json=payload)
    if r.status_code in [200, 201]:
        print('  OK ' + filename + ' subido')
    else:
        print('  Error ' + filename + ': ' + str(r.status_code))

def update_history(all_groups, token, user, repo):
    today = datetime.now().strftime('%Y-%m-%d')
    history, _ = get_github_file('history.json', token, user, repo)
    if history is None: history = []
    history = [h for h in history if h['date'] != today]
    entry = {
        'date': today,
        'scores': {g['group']: g['score'] for g in all_groups},
        'ranks':  {g['group']: i+1 for i, g in enumerate(all_groups)},
    }
    history.append(entry)
    return sorted(history, key=lambda x: x['date'])[-7:]

print('OK Codigo cargado')
print('Recuerda poner GITHUB_TOKEN y ANTHROPIC_KEY arriba.')


In [ ]:
# CELDA 3 — Watchlist personal (~45 seg)
import warnings, json
from datetime import datetime
warnings.filterwarnings('ignore')

if not GITHUB_TOKEN or not ANTHROPIC_KEY:
    print('ERROR: Pon GITHUB_TOKEN y ANTHROPIC_KEY en la Celda 2')
else:
    print('SPY...')
    bc, bv, bh, bl = download_prices(['SPY'], period='1y')
    bs = bc['SPY']
    print('  OK')

    pt = list(set(t for grp in PERSONAL_WATCHLIST.values() for t in grp))
    print('Watchlist (' + str(len(pt)) + ' valores)...')
    cp, vp, hp, lp = download_prices(pt + ['SPY'], period='1y')
    print('  OK')

    print('Analizando...')
    res = analyze_universe(PERSONAL_WATCHLIST, bs, cp, vp, hp, lp)
    gs  = calc_groups(res, is_sp=False)
    ts  = datetime.now().strftime('%d/%m/%Y %H:%M')

    print('Generando analisis Claude...')
    data_tmp = {'timestamp': ts, 'mode': 'Watchlist personal', 'groups': gs, 'values': sorted(res, key=lambda x: x['score'] or 0, reverse=True)}
    analisis = generate_analysis(data_tmp, ANTHROPIC_KEY)

    history = update_history(gs, GITHUB_TOKEN, GITHUB_USER, GITHUB_REPO)

    data = {
        'timestamp': ts, 'mode': 'Watchlist personal',
        'groups': gs,
        'values': sorted(res, key=lambda x: x['score'] or 0, reverse=True),
        'analisis': analisis,
    }

    print('Subiendo a GitHub...')
    upload_to_github('data.json',    json.dumps(data,    ensure_ascii=False), GITHUB_TOKEN, GITHUB_USER, GITHUB_REPO)
    upload_to_github('history.json', json.dumps(history, ensure_ascii=False), GITHUB_TOKEN, GITHUB_USER, GITHUB_REPO)

    print('')
    print('=== RANKING ===')
    for i, g in enumerate(gs):
        bo = ' UP x' + str(g['breakouts']) if g['breakouts'] else ''
        print('#' + str(i+1) + '  ' + str(g['score']) + '  ' + g['group'] + bo)
    print('')
    print('Web actualizada: https://Carolo-III.github.io/scanner-temas')


In [ ]:
# CELDA 4 — S&P 500 completo (~5 min)
import warnings, json, pandas as pd
from datetime import datetime
warnings.filterwarnings('ignore')

if not GITHUB_TOKEN or not ANTHROPIC_KEY:
    print('ERROR: Pon GITHUB_TOKEN y ANTHROPIC_KEY en la Celda 2')
else:
    print('SPY...')
    bc, bv, bh, bl = download_prices(['SPY'], period='1y')
    bs = bc['SPY']
    print('  OK')

    pt = list(set(t for grp in PERSONAL_WATCHLIST.values() for t in grp))
    print('Watchlist (' + str(len(pt)) + ' valores)...')
    cp, vp, hp, lp = download_prices(pt + ['SPY'], period='1y')
    pr  = analyze_universe(PERSONAL_WATCHLIST, bs, cp, vp, hp, lp)
    pgs = calc_groups(pr, is_sp=False)
    print('  OK ' + str(len(pr)) + ' valores')

    print('Lista S&P 500...')
    url = 'https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv'
    df_sp = pd.read_csv(url)
    sbs = {}
    for _, row in df_sp.iterrows():
        tk = row['Symbol'].replace('.', '-')
        if tk not in pt:
            lb = SECTOR_LABELS.get(row['GICS Sector'], row['GICS Sector'])
            sbs.setdefault(lb, []).append(tk)

    sa = list(set(t for grp in sbs.values() for t in grp))
    print('Descargando ' + str(len(sa)) + ' valores en lotes...')
    cs, vs, hs, ls = download_prices(sa, period='1y')
    sr  = analyze_universe(sbs, bs, cs, vs, hs, ls)
    sgs = calc_groups(sr, is_sp=True)
    print('  OK ' + str(len(sr)) + ' valores')

    ar = pr + sr
    ts = datetime.now().strftime('%d/%m/%Y %H:%M')
    all_groups = sorted(pgs + sgs, key=lambda x: x['score'], reverse=True)

    print('Generando analisis Claude...')
    data_tmp = {'timestamp': ts, 'mode': 'S&P500 + Watchlist', 'groups': all_groups, 'values': sorted(ar, key=lambda x: x['score'] or 0, reverse=True)}
    analisis = generate_analysis(data_tmp, ANTHROPIC_KEY)

    history = update_history(all_groups, GITHUB_TOKEN, GITHUB_USER, GITHUB_REPO)

    data = {
        'timestamp': ts, 'mode': 'S&P500 + Watchlist',
        'groups': all_groups,
        'values': sorted(ar, key=lambda x: x['score'] or 0, reverse=True),
        'analisis': analisis,
    }

    print('Subiendo a GitHub...')
    upload_to_github('data.json',    json.dumps(data,    ensure_ascii=False), GITHUB_TOKEN, GITHUB_USER, GITHUB_REPO)
    upload_to_github('history.json', json.dumps(history, ensure_ascii=False), GITHUB_TOKEN, GITHUB_USER, GITHUB_REPO)

    print('')
    print('=== TOP 5 TEMAS ===')
    for i, g in enumerate(all_groups[:5]):
        print('#' + str(i+1) + '  ' + str(g['score']) + '  ' + g['group'])
    bos = [r for r in ar if r['breakout']]
    print('Rupturas: ' + str(len(bos)))
    print('')
    print('Web actualizada: https://Carolo-III.github.io/scanner-temas')
